# Crypto market regime detection: BTC and ETH

Markets do not behave the same way every week. A 2% daily move can be ordinary in one period and a stress signal in another.

This notebook pulls recent daily BTC/USDT and ETH/USDT candles from Binance and clusters market days into regimes using return, volatility, drawdown, and volume-shock features. The point is not price prediction. The point is behavior: when does the market look calm, directional, or stressed?

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOLS = ["BTCUSDT", "ETHUSDT"]
START_DATE = "2024-01-01"
INTERVAL = "1d"
ASSET_DIR = Path("assets")
ASSET_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["font.family"] = "DejaVu Sans"

## 1. Pull recent market data

Binance market-data endpoints do not need an API key. I use daily candles from 2024-01-01 through the latest available day. That keeps the report current enough for a LinkedIn post without depending on a static CSV.

In [ ]:
def fetch_klines(symbol, start_date=START_DATE, interval=INTERVAL):
    start_ms = int(pd.Timestamp(start_date, tz="UTC").timestamp() * 1000)
    rows = []
    while True:
        params = {"symbol": symbol, "interval": interval, "startTime": start_ms, "limit": 1000}
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        batch = response.json()
        if not batch:
            break
        rows.extend(batch)
        next_ms = batch[-1][6] + 1
        if next_ms <= start_ms:
            break
        start_ms = next_ms
        if len(batch) < 1000:
            break
        time.sleep(0.15)

    columns = [
        "open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_asset_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore",
    ]
    df = pd.DataFrame(rows, columns=columns)
    numeric = ["open", "high", "low", "close", "volume", "quote_asset_volume", "trades"]
    df[numeric] = df[numeric].astype(float)
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.date
    df["symbol"] = symbol
    return df[["date", "symbol", "open", "high", "low", "close", "volume", "quote_asset_volume", "trades"]]

market = pd.concat([fetch_klines(symbol) for symbol in SYMBOLS], ignore_index=True)
market["date"] = pd.to_datetime(market["date"])
print(f"Rows: {len(market):,}")
print(f"Date range: {market['date'].min().date()} to {market['date'].max().date()}")
market.tail()

## 2. Feature engineering

A regime model needs market-state features, not just price. For each asset I calculate:

- daily log return
- 7-day and 30-day realized volatility
- drawdown from rolling peak
- volume shock vs 30-day average
- BTC/ETH relative strength

The model is fitted on BTC days, with ETH used as a supporting read on cross-asset behavior.

In [ ]:
def add_features(group):
    group = group.sort_values("date").copy()
    group["log_close"] = np.log(group["close"])
    group["log_return"] = group["log_close"].diff()
    group["vol_7d"] = group["log_return"].rolling(7).std() * np.sqrt(365)
    group["vol_30d"] = group["log_return"].rolling(30).std() * np.sqrt(365)
    group["rolling_peak"] = group["close"].cummax()
    group["drawdown"] = group["close"] / group["rolling_peak"] - 1
    group["volume_ma_30d"] = group["volume"].rolling(30).mean()
    group["volume_shock"] = np.log(group["volume"] / group["volume_ma_30d"])
    return group

features = market.groupby("symbol", group_keys=False).apply(add_features)
wide_close = features.pivot(index="date", columns="symbol", values="close")
relative = np.log(wide_close["ETHUSDT"] / wide_close["BTCUSDT"]).diff().rename("eth_btc_relative_return")
features = features.merge(relative, left_on="date", right_index=True, how="left")
model_df = features.query("symbol == 'BTCUSDT'").dropna().copy()
MODEL_FEATURES = ["log_return", "vol_7d", "vol_30d", "drawdown", "volume_shock", "eth_btc_relative_return"]
print(model_df[MODEL_FEATURES].describe().round(4))

## 3. Regime model

I fit Gaussian Mixture Models with 2 to 5 components and choose the number of regimes with BIC. This is not a full hidden Markov model, but it gives a clean unsupervised regime segmentation and lets us inspect transition behavior afterward.

In [ ]:
X = model_df[MODEL_FEATURES]
model_grid = []
for k in range(2, 6):
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("gmm", GaussianMixture(n_components=k, covariance_type="full", random_state=RANDOM_STATE, n_init=10)),
    ])
    pipe.fit(X)
    scaled = pipe.named_steps["scale"].transform(X)
    labels = pipe.named_steps["gmm"].predict(scaled)
    model_grid.append({
        "k": k,
        "bic": pipe.named_steps["gmm"].bic(scaled),
        "aic": pipe.named_steps["gmm"].aic(scaled),
        "silhouette": silhouette_score(scaled, labels),
        "model": pipe,
    })
selection = pd.DataFrame([{k: v for k, v in row.items() if k != "model"} for row in model_grid])
best = min(model_grid, key=lambda row: row["bic"])
model = best["model"]
scaled = model.named_steps["scale"].transform(X)
model_df["regime_id"] = model.named_steps["gmm"].predict(scaled)
model_df["regime_probability"] = model.named_steps["gmm"].predict_proba(scaled).max(axis=1)
selection

## 4. Name the regimes

The model returns numeric clusters. I label them from their observed behavior: return, volatility, drawdown, and volume shock.

In [ ]:
regime_stats = (
    model_df.groupby("regime_id")
    .agg(
        days=("date", "count"),
        avg_return=("log_return", "mean"),
        vol_30d=("vol_30d", "mean"),
        drawdown=("drawdown", "mean"),
        volume_shock=("volume_shock", "mean"),
        avg_probability=("regime_probability", "mean"),
    )
    .reset_index()
)

def name_regime(row):
    if row["drawdown"] < -0.18 and row["vol_30d"] > regime_stats["vol_30d"].median():
        return "Stress / drawdown"
    if row["avg_return"] > 0 and row["vol_30d"] <= regime_stats["vol_30d"].median():
        return "Calm uptrend"
    if row["avg_return"] > 0 and row["vol_30d"] > regime_stats["vol_30d"].median():
        return "Volatile rally"
    if row["avg_return"] < 0 and row["vol_30d"] > regime_stats["vol_30d"].median():
        return "High-volatility selloff"
    return "Sideways / low momentum"

regime_stats["regime"] = regime_stats.apply(name_regime, axis=1)
regime_map = dict(zip(regime_stats["regime_id"], regime_stats["regime"]))
model_df["regime"] = model_df["regime_id"].map(regime_map)
regime_stats.assign(
    avg_return_pct=regime_stats["avg_return"] * 100,
    avg_drawdown_pct=regime_stats["drawdown"] * 100,
).round(4)

## 5. Charts for the report

In [ ]:
palette = {
    "Calm uptrend": "#16a34a",
    "Volatile rally": "#f59e0b",
    "Stress / drawdown": "#dc2626",
    "High-volatility selloff": "#7f1d1d",
    "Sideways / low momentum": "#2563eb",
}
colors = {name: palette.get(name, "#6b7280") for name in model_df["regime"].unique()}

fig, ax = plt.subplots(figsize=(14, 7))
for regime, chunk in model_df.groupby("regime"):
    ax.scatter(chunk["date"], chunk["close"], s=14, color=colors[regime], label=regime, alpha=0.8)
ax.plot(model_df["date"], model_df["close"], color="#111827", linewidth=0.8, alpha=0.35)
ax.set_title("BTC daily market regimes from recent Binance data")
ax.set_ylabel("BTCUSDT close")
ax.legend(loc="upper left", fontsize=10)
fig.tight_layout()
fig.savefig(ASSET_DIR / "01_btc_regime_timeline.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(data=model_df, x="vol_30d", y="log_return", hue="regime", palette=colors, alpha=0.75, ax=ax)
ax.axhline(0, color="#111827", linestyle="--", linewidth=1)
ax.set_title("Regimes separate return and volatility behavior")
ax.set_xlabel("30-day realized volatility")
ax.set_ylabel("Daily log return")
fig.tight_layout()
fig.savefig(ASSET_DIR / "02_return_volatility_map.png", bbox_inches="tight")
plt.show()

In [ ]:
regime_order = regime_stats.sort_values("days", ascending=False)["regime"].tolist()
transitions = pd.crosstab(model_df["regime"].shift(), model_df["regime"], normalize="index").reindex(index=regime_order, columns=regime_order).fillna(0)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(transitions, annot=True, fmt=".0%", cmap="Blues", ax=ax)
ax.set_title("One-day regime transition matrix")
ax.set_xlabel("Next day")
ax.set_ylabel("Previous day")
fig.tight_layout()
fig.savefig(ASSET_DIR / "03_transition_matrix.png", bbox_inches="tight")
plt.show()
transitions

In [ ]:
risk_table = (
    model_df.groupby("regime")
    .agg(
        days=("date", "count"),
        mean_daily_return=("log_return", "mean"),
        median_vol_30d=("vol_30d", "median"),
        worst_drawdown=("drawdown", "min"),
        avg_probability=("regime_probability", "mean"),
    )
    .reindex(regime_order)
)
risk_table_display = risk_table.assign(
    mean_daily_return=lambda d: d["mean_daily_return"] * 100,
    median_vol_30d=lambda d: d["median_vol_30d"] * 100,
    worst_drawdown=lambda d: d["worst_drawdown"] * 100,
    avg_probability=lambda d: d["avg_probability"] * 100,
).round(2)

fig, ax = plt.subplots(figsize=(11, 4))
ax.axis("off")
table = ax.table(
    cellText=risk_table_display.reset_index().values,
    colLabels=["Regime", "Days", "Mean ret %", "Median vol %", "Worst DD %", "Avg prob %"],
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
ax.set_title("Regime risk summary", pad=18)
fig.tight_layout()
fig.savefig(ASSET_DIR / "04_regime_risk_table.png", bbox_inches="tight")
plt.show()
risk_table_display

In [ ]:
cutoff = model_df["date"].quantile(0.8)
train = model_df[model_df["date"] <= cutoff].copy()
test = model_df[model_df["date"] > cutoff].copy()
train_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("gmm", GaussianMixture(n_components=best["k"], covariance_type="full", random_state=RANDOM_STATE, n_init=10)),
])
train_pipe.fit(train[MODEL_FEATURES])
test_scaled = train_pipe.named_steps["scale"].transform(test[MODEL_FEATURES])
test["regime_id"] = train_pipe.named_steps["gmm"].predict(test_scaled)
test["regime_probability"] = train_pipe.named_steps["gmm"].predict_proba(test_scaled).max(axis=1)

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(model_df["regime_probability"], bins=20, color="#2563eb", alpha=0.55, label="Full sample", ax=ax)
sns.histplot(test["regime_probability"], bins=15, color="#f59e0b", alpha=0.55, label="Out-of-sample tail", ax=ax)
ax.set_title("Regime assignment confidence")
ax.set_xlabel("Maximum regime probability")
ax.legend()
fig.tight_layout()
fig.savefig(ASSET_DIR / "05_assignment_confidence.png", bbox_inches="tight")
plt.show()
print(f"Out-of-sample tail starts after {cutoff.date()}")
print(f"Median tail assignment probability: {test['regime_probability'].median():.2f}")

## 6. Readout

The regimes are useful because they separate market behavior rather than trying to forecast the next candle. The most postable chart is the BTC timeline with regime colors. The serious part is the transition matrix and the risk table: they show persistence, drawdown, and volatility by state.

This is still an unsupervised model. It should be treated as market structure analysis, not trading advice.

In [ ]:
latest = model_df.sort_values("date").iloc[-1]
summary = {
    "data_start": str(model_df["date"].min().date()),
    "data_end": str(model_df["date"].max().date()),
    "btc_rows": int(len(model_df)),
    "selected_regimes": int(best["k"]),
    "latest_regime": latest["regime"],
    "latest_regime_probability": float(latest["regime_probability"]),
    "latest_btc_close": float(latest["close"]),
    "median_assignment_probability": float(model_df["regime_probability"].median()),
    "tail_median_assignment_probability": float(test["regime_probability"].median()),
}
Path("results_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary